# Semester Planning Development

This notebook documents the development of the chatbot's semester planning conversation flow. We describe the changes made to the original output structure, then test three planning scenarios end-to-end.

In [1]:
# ── Setup ──
from src.config import BASE_MODEL, MY_MODEL
from src.chat import Chatbot

def fresh_bot():
    bot = Chatbot()
    bot.update_profile_from_form(
        major_id="6-4",
        courses_str="18.01, 18.02, 8.01, 8.02, 5.111, 7.012, 6.100A, 6.100B",
        year="sophomore",
        semesters_left=5,
    )
    bot.profile.next_is_fall = True
    return bot

def summarize_response(response):
    """Summarize long bot responses for history to prevent format mimicking."""
    if len(response) > 500:
        lines = response.split("\n")
        summary_parts = []
        for line in lines[:3]:
            if line.strip():
                summary_parts.append(line.strip())
        if "SUGGESTED PLANS" in response or "SCORED" in response:
            summary_parts.append("[Presented scored candidates and suggested semester plans.]")
        elif "FINAL" in response:
            summary_parts.append("[Presented final semester plan.]")
        summary_parts.append("[Full response truncated for history.]")
        return " ".join(summary_parts)
    return response

def run_turn(bot, message, history):
    print(f"USER: {message}")
    response = bot.get_response(message, history)
    print(f"\nBOT: {response}")
    plan = bot.profile.semester_plan
    if plan.active:
        print(f"\n  [State] stage={plan.stage}")
        print(f"  [State] planned={plan.planned_courses} ({plan.planned_units}u)")
        if plan.scorer:
            w = plan.scorer.dim_weights
            print(f"  [State] weights: N={w['necessity']:.2f} I={w['interest']:.2f} F={w['feasibility']:.2f}")
            print(f"  [State] profile: {plan._active_profile}")
        if plan.suggested_plans:
            print(f"  [State] available plans: {[p['label'] for p in plan.suggested_plans]}")
    print(f"\n{'━' * 80}\n")
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": summarize_response(response)})
    return history

print(f"Model: {MY_MODEL or BASE_MODEL}")
print("Ready.")

Model: meta-llama/Llama-3.1-8B-Instruct
Ready.


---
## Changes to Semester Planning Output

### Problems with Original Output

The original output structure had several issues that limited its usefulness for semester planning. The most fundamental was a routing problem: asking the chatbot to "plan my semester" bypassed the scored decision model entirely and fell back to a greedy bin-packing planner that simply filled courses by priority until the unit cap was reached. When the scored flow did activate, the LLM was responsible for constructing semester plans itself, which led to invalid combinations — such as including both 18.C06 and 18.06 (which satisfy the same Linear Algebra requirement) — and inconsistent course counts, sometimes recommending five courses plus vague suggestions like "take a HASS course" rather than a concrete four-course plan. The course-level output was also weak: reasons were repetitive ("good fit for your interests in machine learning and AI" appeared for nearly every course regardless of content), scores were shown as raw decimals (0.514) rather than an interpretable scale, and the "lighter load" plan option wasn't actually lighter because the system only considered unit counts rather than estimated hours per week from course evaluations. Finally, the conversational flow was suboptimal — the chatbot jumped straight to recommendations without first asking about interests or constraints, meaning the scoring model was operating on default values, and follow-up questions were generic ("what do you think?") rather than targeting specific tradeoffs between the suggested plans.

### Two-Turn Flow

- **Turn 1 (initiated stage):** Shows a requirements summary (courses remaining, critical prerequisite chains, GIR gaps) and asks a targeted follow-up question based on what information is missing (interests, workload constraints, or specific courses in mind). No courses suggested yet.
- **Turn 2 (gathering_prefs stage):** After the student responds, the system scores all feasible candidates and presents recommendations plus pre-built plans.

### Improved Scorecard

- Scores converted to 1-5 scale (not raw decimals)
- Hours/week shown per course from course evaluation data
- Course descriptions included so LLM can write differentiated reasons
- Key facts (critical path, scarcity, double-counting, CI-M) per course
- Prompt instructs LLM to differentiate reasons, not repeat the same one

### Python-Built Plans

Plans are constructed by Python, not the LLM, since plan composition requires constraint satisfaction (unit caps, no duplicate requirement groups) that language models handle unreliably. The system builds up to four plans:
- **Requirements-focused:** Top 4 courses by overall model score
- **Interest-oriented:** Top 4 courses re-sorted by interest dimension score
- **Lighter load:** 4 courses with the fewest total hours/week
- **Technical heavy:** 4 major requirement courses sorted by necessity

Each plan enforces no duplicate requirement groups and stays under the unit cap. A backtracking mechanism handles cases where an oversized course (e.g., a 15-unit lab) blocks the fourth slot.

### Plan Selection

When the student picks a plan ("I like the lighter load plan"), the system detects the selection by matching on plan labels, plan numbers ("plan 2"), and descriptive keywords ("the easy one"). All courses from the selected plan are added to the session and the system transitions to finalizing.

### Intent Routing Fixes

- "Help me plan my semester" now routes to the scored planning flow instead of the old greedy planner
- When a semester plan is already active, follow-up messages are handled by the planning handler first (before intent detection), preventing keywords like "plan" from resetting the flow
- When the student mentions specific course IDs alongside planning keywords, the system routes to `SEMESTER_BUILD` (add courses and suggest remaining) rather than `PLANNING` (start from scratch)

### History Summarization

Long bot responses (scored candidate lists, plan presentations) are summarized before being added to conversation history. This prevents the LLM from mimicking the format of previous responses when the stage has changed (e.g., repeating the full scored list when it should be finalizing a plan).

---
## Scenario A: Plan From Scratch

The student has no courses in mind. Tests the full flow: initiated → gathering_prefs → finalizing.

**Expected:** Turn 1 asks about interests/constraints. Turn 2 shows scored candidates + plans. Turn 3 finalizes the selected plan.

### Turn 1: Initiate Planning

In [2]:
bot_a = fresh_bot()
history_a = []
history_a = run_turn(bot_a, "Help me plan my fall semester.", history_a)

USER: Help me plan my fall semester.
[DEBUG ROUTING] intent=planning, plan.active=False, plan.stage=None
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=0, priority=False

BOT: To plan your fall semester, let's confirm your profile: You're a sophomore majoring in 6-4 (Artificial Intelligence and Decision Making) with 5 semesters remaining. You've completed the following courses: 18.01, 18.02, 8.01, 8.02, 5.111, 7.012, 6.100A, and 6.100B.

The most urgent items for your fall semester are the critical prerequisite chain: 6.1200 → 6.1210 → 6.1220 (3 semesters). Since 6.1200 is the starting point, it's essential to take it this fall.

Additionally, you have open requirement groups for Linear Algebra, Probability/Inference, Centers - Data-centric, and Centers - Model-centric. You also need two more CI-M courses and have outstanding GIRs, including REST, Institute Lab, HASS, and CI-H.

To help plan your fall semester effectively, can you tell me about your

### Turn 2: State Preferences

In [3]:
history_a = run_turn(bot_a,
    "I'm interested in machine learning and AI. I also have a UROP this semester so I don't want to overload.",
    history_a
)

USER: I'm interested in machine learning and AI. I also have a UROP this semester so I don't want to overload.
[DEBUG ROUTING] intent=general, plan.active=True, plan.stage=initiated
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=0, priority=False
[DEBUG get_response] stage=initiated, valid_ids=[]
[DEBUG get_response] suggested_plans=0 plans
[DEBUG get_response] plan labels: []
[DEBUG get_response] selected_plan=None
[DEBUG get_response] msg_lower: i'm interested in machine learning and ai. i also have a urop this semester so i
[DEBUG _build_semester_plans] major_pool=10, hass_pool=0, max_units=48
  major: 6.4120 req=Centers - Human-centric hrs=8
  major: 6.C571 req=Centers - Decision-centric hrs=11
  major: 6.4210 req=Application CI-M hrs=12
  major: 18.C06 req=Required hrs=11
  major: 6.1200 req=Required hrs=11
[DEBUG] Plan A: 4 courses, 48u, 38hrs — ['6.4120', '6.C571', '18.C06', '18.06']
[DEBUG] Plan B: 4 courses, 48u, 38hrs — ['6.4120', '18.06',

### Turn 3: Select a Plan

In [4]:
history_a = run_turn(bot_a,
    "I like the lighter load plan. Let's go with that.",
    history_a
)

USER: I like the lighter load plan. Let's go with that.
[DEBUG ROUTING] intent=planning, plan.active=True, plan.stage=gathering_prefs
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=4, priority=True
[DEBUG get_response] stage=gathering_prefs, valid_ids=[]
[DEBUG get_response] suggested_plans=4 plans
[DEBUG get_response] plan labels: ['Requirements-focused', 'Interest-oriented', 'Lighter load', 'Technical heavy']
[DEBUG get_response] selected_plan=YES: Lighter load
[DEBUG get_response] msg_lower: i like the lighter load plan. let's go with that.

BOT: FINAL SEMESTER PLAN (48u, ~37 hrs/wk total):

  1. 6.4120 — Computational Cognitive Science (12u, ~8 hrs/wk) 
     - Fulfills: Centers - Human-centric
  2. 18.06 — Linear Algebra (12u, ~8 hrs/wk) 
     - Fulfills: Linear Algebra
  3. 6.3000 — Signal Processing (12u, ~10 hrs/wk) 
     - Fulfills: Centers - Model-centric
  4. 6.C571 — Optimization Methods (12u, ~11 hrs/wk) 
     - Fulfills: Centers - Decis

---
## Scenario B: Pick Courses From Scored List

The student plans from scratch but instead of selecting a pre-built plan, picks 1-2 specific courses from the scored list at Turn 2. The system should add those courses and show scored candidates for the remaining slots.

**Expected:** Turn 1 asks about preferences. Turn 2 shows scored list + plans. Turn 3 adds picked courses, shows remaining slot candidates. Turn 4 fills remaining and finalizes.

### Turn 1: Initiate Planning

In [5]:
bot_b = fresh_bot()
history_b = []
history_b = run_turn(bot_b, "Help me plan my fall semester.", history_b)

USER: Help me plan my fall semester.
[DEBUG ROUTING] intent=planning, plan.active=False, plan.stage=None
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=0, priority=False

BOT: You're a sophomore majoring in 6-4 (Artificial Intelligence and Decision Making), and you have 5 semesters remaining. You've taken the following courses: 18.01, 18.02, 8.01, 8.02, 5.111, 7.012, 6.100A, 6.100B, and you still need to complete the following required courses: 6.1010, 6.1200, 6.1210, 18.C06.

To plan your fall semester, I'd like to highlight a few urgent items. You have a critical prerequisite chain: 6.1200 → 6.1210 → 6.1220 (3 semesters). Additionally, you still need to complete the Linear Algebra, Probability/Inference, and Centers - Data-centric requirements.

Before we start planning your courses, can you tell me what topics or career directions interest you the most? For example, are you interested in machine learning, systems, theory, or quant finance? This w

### Turn 2: State Preferences

In [6]:
history_b = run_turn(bot_b,
    "I want to focus on AI and make progress on requirements. Moderate workload is fine.",
    history_b
)

USER: I want to focus on AI and make progress on requirements. Moderate workload is fine.
[DEBUG ROUTING] intent=requirements, plan.active=True, plan.stage=initiated
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=0, priority=False
[DEBUG get_response] stage=initiated, valid_ids=[]
[DEBUG get_response] suggested_plans=0 plans
[DEBUG get_response] plan labels: []
[DEBUG get_response] selected_plan=None
[DEBUG get_response] msg_lower: i want to focus on ai and make progress on requirements. moderate workload is fi
[DEBUG _build_semester_plans] major_pool=10, hass_pool=0, max_units=48
  major: 6.C571 req=Centers - Decision-centric hrs=11
  major: 6.4210 req=Application CI-M hrs=12
  major: 18.C06 req=Required hrs=11
  major: 6.1200 req=Required hrs=11
  major: 6.4120 req=Centers - Human-centric hrs=8
[DEBUG] Plan A: 4 courses, 48u, 40hrs — ['6.C571', '18.C06', '6.4120', '6.3000']
[DEBUG] Plan B: 4 courses, 48u, 38hrs — ['6.4120', '18.06', '6.C571', '6.1

### Turn 3: Pick 2 Courses From the List

In [7]:
history_b = run_turn(bot_b,
    "I'll take 6.4120 and 18.C06 for sure. What should I fill the other two slots with?",
    history_b
)

USER: I'll take 6.4120 and 18.C06 for sure. What should I fill the other two slots with?
[DEBUG ROUTING] intent=course_lookup, plan.active=True, plan.stage=gathering_prefs
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=3, priority=True
[DEBUG get_response] stage=gathering_prefs, valid_ids=['6.4120', '18.C06']
[DEBUG get_response] suggested_plans=3 plans
[DEBUG get_response] plan labels: ['Requirements-focused', 'Interest-oriented', 'Lighter load']
[DEBUG get_response] selected_plan=None
[DEBUG get_response] msg_lower: i'll take 6.4120 and 18.c06 for sure. what should i fill the other two slots wit

BOT: I'm currently busy — please try again in a few seconds. (Free-tier model gets overloaded sometimes.)

  [State] stage=suggesting
  [State] planned=['6.4120', '18.C06'] (24u)
  [State] weights: N=0.86 I=0.04 F=0.11
  [State] profile: requirements_optimizer
  [State] available plans: ['Requirements-focused', 'Interest-oriented', 'Lighter load']

━━━━━━

### Turn 4: Pick Remaining Courses

In [8]:
# Check what stage we're in and what's planned
plan_b = bot_b.profile.semester_plan
print(f"Before Turn 4: stage={plan_b.stage}, planned={plan_b.planned_courses}")
print()

history_b = run_turn(bot_b,
    "Add 6.C571 and 6.3000 to my plan.",
    history_b
)

Before Turn 4: stage=suggesting, planned=['6.4120', '18.C06']

USER: Add 6.C571 and 6.3000 to my plan.
[DEBUG ROUTING] intent=semester_build, plan.active=True, plan.stage=suggesting
[DEBUG ROUTING] planned_courses=['6.4120', '18.C06'], planned_units=24
[DEBUG ROUTING] suggested_plans=3, priority=True
[DEBUG get_response] stage=suggesting, valid_ids=['6.C571', '6.3000']
[DEBUG get_response] suggested_plans=3 plans
[DEBUG get_response] plan labels: ['Requirements-focused', 'Interest-oriented', 'Lighter load']
[DEBUG get_response] selected_plan=None
[DEBUG get_response] msg_lower: add 6.c571 and 6.3000 to my plan.

BOT: I'm currently busy — please try again in a few seconds. (Free-tier model gets overloaded sometimes.)

  [State] stage=finalizing
  [State] planned=['6.4120', '18.C06', '6.C571', '6.3000'] (48u)
  [State] weights: N=0.86 I=0.04 F=0.11
  [State] profile: requirements_optimizer
  [State] available plans: ['Requirements-focused', 'Interest-oriented', 'Lighter load']

━━━━━━━━━

**Scenario B Evaluation:**

*[Fill in after running:]*
- Turn 3: Did it add the 2 courses and show remaining candidates? *[Y/N]*
- Turn 3: Stage transitioned to suggesting? *[Y/N]*
- Turn 4: Did it finalize with all 4 courses? *[Y/N]*
- Final plan courses: *[list]*

---
## Scenario C: Partial Plan (Start With Courses)

The student arrives with 3 courses already chosen and asks for help picking the rest. Tests the SEMESTER_BUILD entry point: skips initiated/gathering_prefs and goes directly to suggesting.

**Expected:** Turn 1 adds the 3 courses, shows scored candidates for the 1 remaining slot. Turn 2 picks a course and finalizes.

### Turn 1: State Existing Courses

In [9]:
bot_c = fresh_bot()
history_c = []
history_c = run_turn(bot_c,
    "I'm planning to take 6.1200, 6.1010, and 18.C06 next semester. What should I take as a fourth class? I'm interested in AI and want something not too heavy.",
    history_c
)

USER: I'm planning to take 6.1200, 6.1010, and 18.C06 next semester. What should I take as a fourth class? I'm interested in AI and want something not too heavy.
[DEBUG ROUTING] intent=semester_build, plan.active=False, plan.stage=None
[DEBUG ROUTING] planned_courses=[], planned_units=0
[DEBUG ROUTING] suggested_plans=0, priority=False

BOT: Based on the provided data, I recommend the following:

**Top Recommendation:** 11.002 — Making Public Policy (12u, ~8 hrs/wk)  Score: 5/5
This course is a required course for the Planning major and is only offered in the fall. It has a high rating of 5.8/7 and is a good fit for the student's interests in AI, as it explores how government policies are shaped and implemented.

**Runner-up:** 11.001 — Introduction to Urban Design and Development (12u, ~6 hrs/wk)  Score: 5/5
This course is also a required course for the Planning major and has a high rating of 6.2/7. It surveys the ideas of various people who have addressed urban planning and developme

### Turn 2: Pick the Recommended Course

In [10]:
# Check state and pick a course
plan_c = bot_c.profile.semester_plan
print(f"Before Turn 2: stage={plan_c.stage}, planned={plan_c.planned_courses}")
if plan_c.feasible_candidates:
    top_cid = plan_c.feasible_candidates[0]["course_id"]
    print(f"Top candidate: {top_cid}")
    print()
    history_c = run_turn(bot_c, f"Let's add {top_cid}.", history_c)
else:
    print("No feasible candidates — trying explicit pick")
    history_c = run_turn(bot_c, "Add 6.4120 to my plan.", history_c)

Before Turn 2: stage=suggesting, planned=['6.1200', '6.1010', '18.C06']
Top candidate: 11.001

USER: Let's add 11.001.
[DEBUG ROUTING] intent=course_lookup, plan.active=True, plan.stage=suggesting
[DEBUG ROUTING] planned_courses=['6.1200', '6.1010', '18.C06'], planned_units=36
[DEBUG ROUTING] suggested_plans=0, priority=False
[DEBUG get_response] stage=suggesting, valid_ids=['11.001']
[DEBUG get_response] suggested_plans=0 plans
[DEBUG get_response] plan labels: []
[DEBUG get_response] selected_plan=None
[DEBUG get_response] msg_lower: let's add 11.001.

BOT: **Final Semester Plan:**

1. 6.1200 - Mathematics for Computer Science (12u, ~11 hrs/wk) - Planned Elective
2. 6.1010 - Fundamentals of Programming (12u, ~11 hrs/wk) - Planned Elective
3. 18.C06 - Linear Algebra and Optimization (12u, ~11 hrs/wk) - Planned Elective
4. 11.001 - Introduction to Urban Design and Development (12u, ~6 hrs/wk) - Required Course

**Total Units:** 48
**Estimated Hours per Week:** ~39 hrs/wk

Are you happy